In [2]:
! pip install --upgrade plotly
! pip install dash-bootstrap-components

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 24.2 MB/s eta 0:00:00
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.4/202.4 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.0/228.0 kB 17.1 MB/s eta 0:00:00
  Created wheel for stringcase: filename=stringcase-1.2.0-py3-none-any.whl size=3568 sha256=0e2aef16f06aafd4f8fb983d8056af6d4d00965cd1e4a076ccce266c4b9b6810
  Stored in directory: /root/.cache/pip/wheels/b4/33/6d/d0820be98063da218c3206fbad2381cd2db3fbb1a0f0d254b5
Successfully built stringcase
  Attempting uninstall: Werkzeug
    Found existing installation: Werkzeug 3.1.3
    Uninstalling Werkzeug-3.1.3:

Перед тем как делать слайдеры в dash я хочу создать функцию для этих слайдеров, чтобы оно подгоняло marks при необходимости в зависимости от min и max значений. Просто будем разбивать ее на 5 равных частей.

In [3]:
def marks_split_to_5(min_val, max_val):
    step = (max_val - min_val) / 4
    marks = {int(min_val + i * step): f'{int(min_val + i * step)}' for i in range(5)}
    return marks

In [9]:
min_rent = 1000
max_rent = 3500

min_wage = 20000
max_wage = 45000

min_capital = 500000
max_capital = 5000000

min_equipment = 100000
max_equipment = 300000

min_rooms = 5
max_rooms = 15

avg_check = 300

In [ ]:
import dash
from dash import dcc, html, Input, Output
import plotly.graph_objs as go
import numpy as np


#нам вроде не рассказывали про слайдеры, поэтому я вот отсюда взяла
#https://github.com/DashBookProject/Plotly-Dash/blob/master/Bonus-Content/Components/sliders.md
#https://dash.plotly.com/dash-core-components/slider
#https://habr.com/ru/articles/502958/
#из последней сссылки я взяла основную идею визуализации через go.Figure(), другие варианты updаte показались мне сложнее

app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("Финансовая модель капсульного караоке"),

    html.Label("Аренда помещения за кв.м. (руб/месяц)"),
    dcc.Slider(min=min_rent, max=max_rent, step=250, value=(min_rent+ max_rent)/2, marks = marks_split_to_5(min_rent, max_rent), id='mean_rent'),

    html.Label("Заработная плата (руб/месяц)"),
    dcc.Slider(min=min_wage, max=max_wage, step=250, value=(min_wage+ max_wage)/2, marks = marks_split_to_5(min_wage, max_wage), id='wage'),

    html.Label("Стартовый капитал (руб)"),
    dcc.Slider(min=min_capital, max=max_capital, step=100000, value=(min_capital+ max_capital)/2, marks = marks_split_to_5(min_capital, max_capital), id='capital'),

    html.Label("Стоимость оборудования (руб/комната)"),
    dcc.Slider(min=min_equipment, max=max_equipment, step=10000, value=(min_equipment+ max_equipment)/2, marks = marks_split_to_5(min_equipment, max_equipment), id='equipment'),

    html.Label("Количество комнат"),
    dcc.Slider(min=min_rooms, max=max_rooms, step=1, value=(min_rooms+ max_rooms)/2, marks = marks_split_to_5(min_rooms, max_rooms), id='rooms'),

    dcc.Graph(id='finance-graph')
])

@app.callback(
    Output('finance-graph', 'figure'),
    [Input('mean_rent', 'value'),
     Input('wage', 'value'),
     Input('capital', 'value'),
     Input('equipment', 'value'),
     Input('rooms', 'value')]
)

def update_graph(mean_rent, wage, capital, equipment, rooms):

    '''
    rent - будем считать, что размер одной капсулы это 4 кв метра и пусть нам надо еще 12 кв.м. на места общего пользования
    capital - будем считать что мы взяли кредит равный стартовому капиталу и ходим его выплатить за 12 месяцев (пусть кредит на 10 лет при 10 процентов годовых)
    monthly_investment_payment - выплаты по этому кредиту
    start_capital - итог нашего бюджета без ежемесячных расходов
    monthly_expenses - те самые расходы, считаем что у нас 3 работника

    guests - будем рандомить число гостей, пусть на одну комнату от 1 до 4 гостей и будем считать, что заняты будут от 2 до всех комнат на рандом
            хотя такая рандом не очень правдоподобен, но все таки схема примерная. При этом у нас будет условно "4 волн" такого рандома, на 28 рабочих дней в месяце

            чтобы было более правдоподобно первые 5 месяцев кол-во гостей будет расти экпоненциально до среднего кол-ва гостей, а потом уже будет рандомиться
    '''

    rent = mean_rent * (rooms + 3) * 4

    monthly_investment_payment = capital*(1.3) / 12
    start_capital = capital - equipment * rooms
    current_capital = start_capital

    monthly_expenses = rent + 3 * wage

    months = np.arange(1, 25)
    finances = [start_capital]

    mean_guests = 2 * ((rooms + 1) // 2 + 2) * 4 * 20
    guests_list = [0]

    for month in months:

      if month <= 12:
        current_capital -= monthly_investment_payment
        guests = int(mean_guests * (1 - np.exp(-month/2))) #деление на 2 чтобы плавнее рост был
      else:
        guests = sum(np.random.randint(4, rooms + 1, size= 2 * 4*20))

      guests_list.append(guests)

      income = guests * avg_check

      current_capital += income - monthly_expenses

      finances.append(current_capital)


    fig = go.Figure()
    fig.add_trace(go.Scatter(
    x=months,
    y=[capital] * len(months),
    mode='lines',
    name='Начальный капитал',
    line=dict(color='red')
    ))
    fig.add_trace(go.Scatter(
        x=months,
        y=finances,
        mode='lines+markers',
        name='Финановый тренд',
        line=dict(color='blue')
    ))
    fig.update_layout(
        title="Финановый тренд за 2 года",
        xaxis_title="Месяц",
        yaxis_title="Капитал (руб)")
    return fig


if __name__ == '__main__':
    app.run(debug=True)

<IPython.core.display.Javascript object>

In [53]:
import dash
from dash import dcc, html, Input, Output
import plotly.graph_objs as go
import numpy as np


#нам не рассказывали про слайдеры, поэтому я вот отсюда взяла
#https://github.com/DashBookProject/Plotly-Dash/blob/master/Bonus-Content/Components/sliders.md
#https://dash.plotly.com/dash-core-components/slider
#https://habr.com/ru/articles/502958/
#из последней сссылки я взяла основную идею визуализации через go.Figure(), другие варианты update показались мне заметно сложнее

app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("Финансовая модель капсульного караоке"),

    html.Label("Аренда помещения за кв.м. (руб/месяц)"),
    dcc.Slider(min=min_rent, max=max_rent, step=250, value=(min_rent+ max_rent)/2, marks = marks_split_to_5(min_rent, max_rent), id='mean_rent'),

    html.Label("Заработная плата (руб/месяц)"),
    dcc.Slider(min=min_wage, max=max_wage, step=250, value=(min_wage+ max_wage)/2, marks = marks_split_to_5(min_wage, max_wage), id='wage'),

    html.Label("Стартовый капитал (руб)"),
    dcc.Slider(min=min_capital, max=max_capital, step=100000, value=(min_capital+ max_capital)/2, marks = marks_split_to_5(min_capital, max_capital), id='capital'),

    html.Label("Стоимость оборудования (руб/комната)"),
    dcc.Slider(min=min_equipment, max=max_equipment, step=10000, value=(min_equipment+ max_equipment)/2, marks = marks_split_to_5(min_equipment, max_equipment), id='equipment'),

    html.Label("Количество комнат"),
    dcc.Slider(min=min_rooms, max=max_rooms, step=1, value=(min_rooms+ max_rooms)/2, marks = marks_split_to_5(min_rooms, max_rooms), id='rooms'),

    dcc.Graph(id='finance-graph')
])

@app.callback(
    Output('finance-graph', 'figure'),
    [Input('mean_rent', 'value'),
     Input('wage', 'value'),
     Input('capital', 'value'),
     Input('equipment', 'value'),
     Input('rooms', 'value')]
)

def update_graph(mean_rent, wage, capital, equipment, rooms):

    '''
    rent - будем считать, что размер одной капсулы это 4 кв метра и пусть нам надо еще 12 кв.м. на места общего пользования
    capital - будем считать что мы взяли кредит равный стартовому капиталу и ходим его выплатить за 12 месяцев (пусть кредит на 10 лет при 10 процентов годовых)
    monthly_investment_payment - выплаты по этому кредиту
    start_capital - итог нашего бюджета без ежемесячных расходов
    monthly_expenses - те самые расходы, считаем что у нас 3 работника

    guests - будем рандомить число гостей, пусть на одну комнату от 1 до 4 гостей и будем считать, что заняты будут от 2 до всех комнат на рандом
            хотя такая рандом не очень правдоподобен, но все таки схема примерная. При этом у нас будет условно "4 волн" такого рандома, на 28 рабочих дней в месяце

            чтобы было более правдоподобно первые 5 месяцев кол-во гостей будет расти экпоненциально до среднего кол-ва гостей, а потом уже будет рандомиться
    '''

    rent = mean_rent * (rooms + 3) * 4

    monthly_investment_payment = capital*(1.3) / 12
    start_capital = capital - equipment * rooms

    monthly_expenses = rent + 3 * wage

    months = np.arange(1, 25)
    finances = [start_capital]
    current_capital = capital

    mean_guests = 2 * ((rooms + 1) // 2 + 2) * 4 * 20
    guests_list = [0]

    for month in months:

      if month <= 12:
        current_capital -= monthly_investment_payment
        guests = int(mean_guests * (1 - np.exp(-month/2)))
      else:
        guests = sum(np.random.randint(4, rooms + 1, size= 2 * 4*20))

      guests_list.append(guests)

      income = guests * avg_check

      current_capital += income - monthly_expenses

      finances.append(current_capital)


    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=months,
        y=guests_list,
        mode='lines+markers',
        name='Финановый тренд',
        line=dict(color='blue')
    ))
    fig.update_layout(
        title="Посещаемость заведения за 2 года",
        xaxis_title="Месяц",
        yaxis_title="Кол-во людей")
    return fig


if __name__ == '__main__':
    app.run(debug=True)

<IPython.core.display.Javascript object>